# ⚔️ B4　Boss 戰：小小點餐系統
**Python 冒險之旅 2026**　｜　Day 4（09/03 四）🏔️ 函式之島　｜　Boss 戰　｜　🏅 200 XP

📖 對應教科書：第 7–8 章綜合


### 🎯 這一關你會學到
- 用函式 + 字典設計一個可重複使用的小系統

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "B4"
_SALT = "python-quest-2026-datama"
_TASKS = ["B4-1", "B4-2", "B4-3", "B4-4", "B4-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B4_1(run):
    out, ns = run()
    if not callable(ns.get("show_menu")): return (False, "要定義 show_menu。")
    lines = 行列表(out)
    return (len(lines) == 6 and lines[0].startswith("=== 菜單") and "珍珠奶茶" in lines[1] and lines[1].endswith("  60"), "應該印出標題加 5 行菜單，價格靠右佔 4 格。")
任務定義("B4-1", _check_B4_1, 提示="menu.items() 可以同時拿到鍵與值。")

def _check_B4_2(run):
    out, ns = run()
    f = ns.get("add_item")
    if not callable(f): return (False, "要定義 add_item。")
    if ns.get("order") != {'珍珠奶茶': 3, '雞排': 1}: return (False, f"order 應該是 {{'珍珠奶茶': 3, '雞排': 1}}，現在是 {ns.get('order')}。")
    o = {}; f(o, 'a', 1); f(o, 'a', 4); f(o, 'b', 2)
    return (o == {'a': 5, 'b': 2}, "重複加入同一品項要累加數量。")
任務定義("B4-2", _check_B4_2, 提示="order[name] += qty 或 order[name] = qty。")

def _check_B4_3(run):
    out, ns = run()
    t, d, m = ns.get("total"), ns.get("discount"), ns.get("menu")
    if not (callable(t) and callable(d)): return (False, "要定義 total 與 discount。")
    if t({'珍珠奶茶': 3, '雞排': 1, '鹹酥雞': 4}, m) != 575: return (False, "總額應該是 575。")
    if d(575) != 517: return (False, "575 打 9 折應該是 517（int）。")
    if d(1200) != 960: return (False, "1200 打 8 折應該是 960。")
    return (d(300) == 300, "300 不打折。")
任務定義("B4-3", _check_B4_3, 提示="amount += menu[name] * qty；折扣用 if/elif 判斷後 return int(...)。")

def _check_B4_4(run):
    out, ns = run()
    need = ["珍珠奶茶x3=180", "雞排x1=75", "鹹酥雞x4=320", "合計575元", "折後517元"]
    missing = [n for n in need if not 出現(out, n)]
    return (not missing, f"收據少了：{missing}")
任務定義("B4-4", _check_B4_4, 提示="小計 = menu[name] * qty；折後 = discount(t)。")

def _check_B4_5(run):
    out, ns = run("雞排 2", "牛排 1", "紅茶 3", "結帳")
    if not 出現(out, "沒有這個品項"): return (False, "輸入不存在的 牛排 要印出 沒有這個品項。")
    if ns.get("order") != {'雞排': 2, '紅茶': 3}: return (False, f"order 應該是 {{'雞排': 2, '紅茶': 3}}，現在是 {ns.get('order')}。")
    return (出現(out, "合計240元"), "合計應該是 240 元。")
任務定義("B4-5", _check_B4_5, 提示="條件 name not in menu。")


## ⚔️ Boss 登場：小小點餐系統
勇者餐廳要用 Python 做點餐系統。菜單用**字典**存，功能用**函式**包裝：

| 功能 | 說明 |
|---|---|
| `show_menu(menu)` | 印出菜單 |
| `add_item(order, name, qty)` | 把品項加入訂單（訂單也是字典：品名 → 數量） |
| `total(order, menu)` | 計算總金額 |
| `discount(amount)` | 滿 500 打 9 折、滿 1000 打 8 折 |
| `receipt(order, menu)` | 印出收據 |

In [ ]:
menu = {'珍珠奶茶': 60, '雞排': 75, '滷肉飯': 45, '紅茶': 30, '鹹酥雞': 80}
print(menu)

### 🎯 任務 B4-1　顯示菜單

定義 `show_menu(menu)`：印出標題 `=== 菜單 ===`，接著每一行 `品名 ... 價格`，品名靠左佔 6 格、價格靠右佔 4 格，例如 `珍珠奶茶    60`。

In [ ]:
# 🎯 任務 B4-1　顯示菜單（請保留這一行）
menu = {'珍珠奶茶': 60, '雞排': 75, '滷肉飯': 45, '紅茶': 30, '鹹酥雞': 80}
def show_menu(menu):
    print("=== 菜單 ===")
    for name, price in ???:
        print(f"{name:<6}{price:>4}")
show_menu(menu)

In [ ]:
檢查("B4-1")   # ◀ 執行這一格，看看任務 B4-1 有沒有過關

### 🎯 任務 B4-2　加入訂單

定義 `add_item(order, name, qty)`：如果 `name` 已在訂單就**累加**數量，否則新增。依序加入 珍珠奶茶×2、雞排×1、珍珠奶茶×1 後印出 `order`。預期 `{'珍珠奶茶': 3, '雞排': 1}`。

In [ ]:
# 🎯 任務 B4-2　加入訂單（請保留這一行）
def add_item(order, name, qty):
    if name in order:
        ???
    else:
        ???
order = {}
add_item(order, '珍珠奶茶', 2)
add_item(order, '雞排', 1)
add_item(order, '珍珠奶茶', 1)
print(order)

In [ ]:
檢查("B4-2")   # ◀ 執行這一格，看看任務 B4-2 有沒有過關

### 🎯 任務 B4-3　計算總額與折扣

定義 `total(order, menu)` 傳回訂單總金額；定義 `discount(amount)`：≥ 1000 打 8 折、≥ 500 打 9 折、否則原價，傳回**整數**。訂單 `{'珍珠奶茶': 3, '雞排': 1, '鹹酥雞': 4}` → 總額 575 → 折後 517。

In [ ]:
# 🎯 任務 B4-3　計算總額與折扣（請保留這一行）
menu = {'珍珠奶茶': 60, '雞排': 75, '滷肉飯': 45, '紅茶': 30, '鹹酥雞': 80}
def total(order, menu):
    amount = 0
    for name, qty in order.items():
        amount += ???
    return amount
def discount(amount):
    ???
order = {'珍珠奶茶': 3, '雞排': 1, '鹹酥雞': 4}
t = total(order, menu)
print(t, discount(t))

In [ ]:
檢查("B4-3")   # ◀ 執行這一格，看看任務 B4-3 有沒有過關

### 🎯 任務 B4-4　列印收據

定義 `receipt(order, menu)` 印出收據：每行 `品名 x數量 = 小計`，最後印 `合計 575 元` 與 `折後 517 元`（可直接使用上一題的 `total`、`discount`；這一格請再定義一次確保可獨立執行）。

In [ ]:
# 🎯 任務 B4-4　列印收據（請保留這一行）
menu = {'珍珠奶茶': 60, '雞排': 75, '滷肉飯': 45, '紅茶': 30, '鹹酥雞': 80}
def total(order, menu):
    return sum(menu[n] * q for n, q in order.items())
def discount(amount):
    if amount >= 1000: return int(amount * 0.8)
    if amount >= 500: return int(amount * 0.9)
    return amount
def receipt(order, menu):
    print("===== 收據 =====")
    for name, qty in order.items():
        print(f"{name} x{qty} = {???}")
    t = total(order, menu)
    print(f"合計 {t} 元")
    print(f"折後 {???} 元")
receipt({'珍珠奶茶': 3, '雞排': 1, '鹹酥雞': 4}, menu)

In [ ]:
檢查("B4-4")   # ◀ 執行這一格，看看任務 B4-4 有沒有過關

### 🎯 任務 B4-5　互動點餐

用 `while True` 讓使用者重複輸入 `品名 數量`（用空格隔開，例如 `雞排 2`），輸入 `結帳` 就結束並印出收據。品名不在菜單時印 `沒有這個品項`。（可沿用前面定義的函式，這一格請完整包含所需函式。）

In [ ]:
# 🎯 任務 B4-5　互動點餐（請保留這一行）
menu = {'珍珠奶茶': 60, '雞排': 75, '滷肉飯': 45, '紅茶': 30, '鹹酥雞': 80}
def add_item(order, name, qty):
    order[name] = order.get(name, 0) + qty
def total(order, menu):
    return sum(menu[n] * q for n, q in order.items())
def discount(amount):
    if amount >= 1000: return int(amount * 0.8)
    if amount >= 500: return int(amount * 0.9)
    return amount
order = {}
while True:
    line = input("請輸入 品名 數量（或輸入 結帳）：")
    if line == "結帳":
        break
    parts = line.split()
    name, qty = parts[0], int(parts[1])
    if ???:
        print("沒有這個品項")
        continue
    add_item(order, name, qty)
t = total(order, menu)
print(f"合計 {t} 元，折後 {discount(t)} 元")

In [ ]:
檢查("B4-5")   # ◀ 執行這一格，看看任務 B4-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 讓 `discount()` 支援「會員再折 20 元」的關鍵字參數 `member=False`。
2. 把訂單用 `datetime` 加上時間戳記，存成字典 `{'time': ..., 'items': order, 'total': t}`。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：📁 L10 檔案與例外處理** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L10_files_exceptions.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/